In [1]:
import os
import numpy as np
import pandas as pd
import json


In [2]:
root_directory = 'datasets/ZooScan77/train'

file_counts_available = {}


for subdir, dirs, files in os.walk(root_directory):
    if subdir == root_directory:
        continue
    class_name = os.path.basename(subdir)
    file_counts_available [class_name] = {'available_samples': len(files)}


In [3]:
stats_df = pd.read_csv('checkpoints/swin_mixup_100_full_contrastive/run_1/stats_df.csv')
df = stats_df[stats_df['Epoch']==1]

median_recall = df['Recall'].median()
median_precision = df['Precision'].median()

num_samples_values = [entry['available_samples'] for entry in file_counts_available.values()]
q1_samples = int(np.percentile(num_samples_values, 25))
median_num_samples = np.median(num_samples_values)
upper_upsampling_bound = int(median_num_samples*1.5)
hard_cap = int(np.percentile(num_samples_values, 90))
print(f"Median number of samples: {median_num_samples}")
print(f"Median Precision: {median_precision:.4f}\nMedian Recall: {median_recall:.4f}")
print(f"Upper upsamling bound: {upper_upsampling_bound}")
print(f"underrepresnted bound: {q1_samples}")
print(f"Hard cap: {hard_cap}")

Median number of samples: 1542.0
Median Precision: 0.8966
Median Recall: 0.9512
Upper upsamling bound: 2313
underrepresnted bound: 416
Hard cap: 11428


In [4]:
root_directory = 'datasets/ZooScan77_010_final/train'

file_counts = {}


for subdir, dirs, files in os.walk(root_directory):
    if subdir == root_directory:
        continue
    class_name = os.path.basename(subdir)
    num_samples = len(files)
    available_samples = file_counts_available [class_name]['available_samples']
    file_counts[class_name] = {'num_samples': num_samples,
                            'Recall': round(df[df['Class Name']== class_name]['Recall'].iloc[0],4),
                            'Precision': round(df[df['Class Name']== class_name]['Precision'].iloc[0],4),
                            'available_samples': available_samples}
    # uppsample
    if  file_counts[class_name]['Recall'] < median_recall and file_counts[class_name]['num_samples'] < upper_upsampling_bound:
        file_counts[class_name]['sample_diff'] = min(upper_upsampling_bound-num_samples, available_samples-num_samples)
    # downsample
    # insure no extremes (like detritus)
    elif num_samples > hard_cap:
        file_counts[class_name]['sample_diff'] = hard_cap - num_samples
    elif  file_counts[class_name]['num_samples'] > upper_upsampling_bound and  file_counts[class_name]['Precision'] < median_precision:
        file_counts[class_name]['sample_diff'] = upper_upsampling_bound-num_samples
    # Ensure Minimum Representation: If metrics are good but samples below the median
    elif num_samples < q1_samples:
        file_counts[class_name]['sample_diff'] = min(q1_samples - num_samples, available_samples-num_samples)
    
    # ok
    else:
        file_counts[class_name]['sample_diff'] = 0



with open('file_counts.json', 'w') as json_file:
    json.dump(file_counts, json_file, indent=4)


Median number of samples: 1542.0
